# Sorting Collections in Java: `Comparable` and `Comparator`
Java provides two standard interfaces to sort objects:

| Feature | `Comparable` (`java.lang`) | `Comparator` (`java.util`) |
| :--- | :--- | :--- |
| **Sorting Type** | **Natural Default Sorting Order** | **Custom / Multiple Attribute Sorting Order** |
| **Method** | `public int compareTo(T other)` | `public int compare(T o1, T o2)` |
| **Modification** | Modifies the original class definition | Implemented in separate helper classes or lambdas |
| **Sorting Call** | `Collections.sort(list)` | `Collections.sort(list, comparator)` or `list.sort(comparator)` |
| **Number of Sorts**| Only 1 sorting sequence | Multiple different sorting sequences |


---
## 1. Natural Sorting Using `Comparable` Interface
Sorts students by `rollNo` naturally:


In [ ]:
import java.util.ArrayList;
import java.util.Collections;
import java.util.List;

class StudentItem implements Comparable<StudentItem> {
    int rollNo;
    String name;
    double marks;

    StudentItem(int rollNo, String name, double marks) {
        this.rollNo = rollNo;
        this.name = name;
        this.marks = marks;
    }

    // Natural ordering: Ascending by rollNo
    @Override
    public int compareTo(StudentItem other) {
        return Integer.compare(this.rollNo, other.rollNo);
    }

    @Override
    public String toString() {
        return "Student{" + rollNo + ", '" + name + "', " + marks + "}";
    }
}

List<StudentItem> students = new ArrayList<>();
students.add(new StudentItem(103, "Charlie", 78.5));
students.add(new StudentItem(101, "Alice", 92.0));
students.add(new StudentItem(102, "Bob", 85.5));

// Natural sort using Comparable
Collections.sort(students);
System.out.println("Naturally Sorted by RollNo (Comparable):");
for (StudentItem s : students) System.out.println("  " + s);


---
## 2. Custom Sorting Using `Comparator` Interface
Allows defining multiple sorting sequences (e.g. by Name, by Marks descending) without altering the original class:


In [ ]:
import java.util.Comparator;

// 1. Separate Comparator class: Sort by Name Alphabetical
class NameComparator implements Comparator<StudentItem> {
    @Override
    public int compare(StudentItem s1, StudentItem s2) {
        return s1.name.compareTo(s2.name);
    }
}

Collections.sort(students, new NameComparator());
System.out.println("\nSorted by Name (NameComparator):");
for (StudentItem s : students) System.out.println("  " + s);

// 2. Anonymous Class implementation of Comparator
Collections.sort(students, new Comparator<StudentItem>() {
    @Override
    public int compare(StudentItem s1, StudentItem s2) {
        return Double.compare(s2.marks, s1.marks); // Descending by marks
    }
});
System.out.println("\nSorted by Marks Descending (Anonymous Comparator):");
for (StudentItem s : students) System.out.println("  " + s);

// 3. Modern Java 8+ Comparator with Lambda & Helper methods
students.sort(Comparator.comparing(s -> s.name)); // lambda
students.sort(Comparator.comparingDouble((StudentItem s) -> s.marks).reversed()); // method chaining
System.out.println("\nSorted with Comparator.comparingDouble().reversed():");
for (StudentItem s : students) System.out.println("  " + s);


---
## 3. Sorting Objects Based on Multiple Attributes (Topic 99)
In production applications, datasets frequently require **multi-tier sorting** (e.g. primary sort by Department; if identical, secondary sort by Salary descending; if identical, tertiary sort by Name):

1. **Modern Java 8+ Chaining (`thenComparing`)**:
   `Comparator.comparing(Getter).thenComparing(Getter).thenComparing(...)`
   - Clean, highly readable, declarative, and completely null-safe.
2. **Traditional Comparator Cascading**:
   Comparing field 1; if `result != 0` return result; otherwise compare field 2.


In [ ]:
import java.util.ArrayList;
import java.util.List;
import java.util.Comparator;

class Candidate {
    String department;
    String name;
    double score;

    Candidate(String dept, String name, double score) {
        this.department = dept;
        this.name = name;
        this.score = score;
    }

    public String getDepartment() { return department; }
    public String getName() { return name; }
    public double getScore() { return score; }

    @Override
    public String toString() {
        return String.format("%-10s | %-10s | %.1f", department, name, score);
    }
}

List<Candidate> candidates = new ArrayList<>();
candidates.add(new Candidate("IT", "Charlie", 85.0));
candidates.add(new Candidate("HR", "Alice", 90.0));
candidates.add(new Candidate("IT", "Bob", 92.5));
candidates.add(new Candidate("HR", "David", 78.0));
candidates.add(new Candidate("IT", "Alice", 85.0)); // same dept and score as Charlie

// Multi-Attribute Sort:
// 1st: Department (A-Z)
// 2nd: Score (Descending: High to Low)
// 3rd: Name (A-Z)
candidates.sort(
    Comparator.comparing(Candidate::getDepartment)
              .thenComparing(Comparator.comparingDouble(Candidate::getScore).reversed())
              .thenComparing(Candidate::getName)
);

System.out.println("Sorted by Department -> Score (Desc) -> Name:");
for (Candidate c : candidates) {
    System.out.println("  " + c);
}
